# Implementing Advantage-Actor Critic (A2C) - 2 pts

In this notebook you will implement Advantage Actor Critic algorithm that trains on a batch of Atari 2600 environments running in parallel. 

Firstly, we will use environment wrappers implemented in file `atari_wrappers.py`. These wrappers preprocess observations (resize, grayscal, take max between frames, skip frames, stack them together, prepares for PyTorch and normalizes to [0, 1]) and rewards. Some of the wrappers help to reset the environment and pass `done` flag equal to `True` when agent dies.
File `env_batch.py` includes implementation of `ParallelEnvBatch` class that allows to run multiple environments in parallel. To create an environment we can use `nature_dqn_env` function.

In [1]:
# !pip install gymnasium==1.0.0
# !pip install ale-py==0.10.2
# !pip install opencv-python
# !pip install gymnasium[other]
# !pip install tensorboardX

In [60]:
import numpy as np
from atari_wrappers import nature_dqn_env
import gymnasium as gym
from atari_wrappers import TensorboardSummaries

nenvs = 30    # change this if you have more than 8 CPU ;)
env = gym.vector.AsyncVectorEnv([lambda: nature_dqn_env("SpaceInvadersNoFrameskip-v4") for _ in range(nenvs)])
env = TensorboardSummaries(env, "spaceinvaders")


n_actions = env.single_action_space.n
obs, info = env.reset()
assert obs.shape == (nenvs, 4, 84, 84)
assert obs.dtype == np.float32

Next, we will need to implement a model that predicts logits of policy distribution and critic value. Use shared backbone. You may use same architecture as in DQN task with one modification: instead of having a single output layer, it must have two output layers taking as input the output of the last hidden layer (one for actor, one for critic). 

Still it may be very helpful to make more changes:
* use orthogonal initialization with gain $\sqrt{2}$ and initialize biases with zeros;
* use more filters (e.g. 32-64-64 instead of 16-32-64);
* use two-layer heads for actor and critic or add a linear layer into backbone;

**Danger:** do not divide on 255, input is already normalized to [0, 1] in our wrappers!

In [61]:
import torch
import torch.nn as nn

N_FRAMES_STACKED = 4

def get_init_conv_weights(x):
    if type(x) == torch.nn.Conv2d:
        torch.nn.init.orthogonal_(x.weight, gain=np.sqrt(2))
        x.bias.data.fill_(0)

class ConvBackbone(nn.Module):
    def __init__(self, c_in: int = N_FRAMES_STACKED) -> None:
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(c_in, 32, kernel_size=8, stride=4), 
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2), 
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1), 
            nn.ReLU(),
            nn.Flatten(),
        ).apply(get_init_conv_weights)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.backbone(x)

        return features

class ModelSharedBackbone(nn.Module):
    def __init__(self, n_actions, inp_size=64 * 7 * 7, hidden_size=512) -> None:
        super().__init__()
        self.actor  = nn.Sequential(
            nn.Linear(inp_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, n_actions)
            
        )
        self.critic = nn.Sequential(
            nn.Linear(inp_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        logits = self.actor(x)
        critic_values = self.critic(x)
        return logits, critic_values

class A2CNet(nn.Sequential):
    '''
    input:
        states - tensor, (batch_size x channels x width x height)
    output:
        logits - tensor, logits of action probabilities for your actor policy, (batch_size x num_actions)
        V - tensor, critic estimation, (batch_size)
    '''
    def __init__(self, c_in: int, n_actions: int) -> None:
        super().__init__()
        self.backbone = ConvBackbone(c_in=c_in)
        self.head = ModelSharedBackbone(n_actions=n_actions)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.as_tensor(x, dtype=torch.float32)
        features = self.backbone(x)
        logits, critic_values = self.head(features)
        return logits, critic_values
    
        

You will also need to define and use a policy that wraps the model. While the model computes logits for all actions, the policy will sample actions and also compute their log probabilities.  `policy.act` should return a **dictionary** of all the arrays that are needed to interact with an environment and train the model.

**Important**: "actions" will be sent to environment, they must be numpy array or list, not PyTorch tensor.

Note: you can add more keys, e.g. it can be convenient to compute entropy right here.

In [62]:
from torch.distributions import Categorical

class Policy:
    def __init__(self, model):
        self.model = model

    def act(self, inputs):
        '''
        input:
            inputs - numpy array, (batch_size x channels x width x height)
        output: dict containing keys ['actions', 'logits', 'log_probs', 'values']:
            'actions' - selected actions, numpy, (batch_size)
            'logits' - actions logits, tensor, (batch_size x num_actions)
            'log_probs' - log probs of selected actions, tensor, (batch_size)
            'values' - critic estimations, tensor, (batch_size)
        '''
        logits, values = self.model(inputs)
        dist = Categorical(logits=logits)
        
        actions = dist.sample()
        
        log_probs = dist.log_prob(actions)
        entropy = dist.entropy()
        return {
            "actions": actions.numpy(),
            "logits": logits,
            "log_probs": log_probs,
            "values": values.squeeze(),
            "entropy": entropy,
        }

Next we will pass the environment and policy to a runner that collects rollouts from the environment. 
The class is already implemented for you.

In [63]:
from runners import EnvRunner

This runner interacts with the environment for a given number of steps and returns a dictionary containing
keys 

* 'observations' 
* 'rewards' 
* 'dones'
* 'actions'
* all other keys that you defined in `Policy`

under each of these keys there is a python `list` of interactions with the environment of specified length $T$ &mdash; the size of partial trajectory, or rollout length. Let's have a look at how it works.

In [82]:
model = A2CNet(N_FRAMES_STACKED, n_actions)
policy = Policy(model)
runner = EnvRunner(env, policy, nsteps=5)

In [83]:
# generates new rollout
trajectory = runner.get_next()

In [84]:
# what is inside
print(trajectory.keys())

dict_keys(['actions', 'logits', 'log_probs', 'values', 'entropy', 'observations', 'rewards', 'dones'])


In [85]:
# Sanity checks
assert 'logits' in trajectory, "Not found: policy didn't provide logits"
assert 'log_probs' in trajectory, "Not found: policy didn't provide log_probs of selected actions"
assert 'values' in trajectory, "Not found: policy didn't provide critic estimations"
assert trajectory['logits'][0].shape == (nenvs, n_actions), "logits wrong shape"
assert trajectory['log_probs'][0].shape == (nenvs,), "log_probs wrong shape"
assert trajectory['values'][0].shape == (nenvs,), "values wrong shape"

for key in trajectory.keys():
    assert len(trajectory[key]) == 5, \
    f"something went wrong: 5 steps should have been done, got trajectory of length {len(trajectory[key])} for '{key}'"

Now let's work with this trajectory a bit. To train the critic you will need to compute the value targets. It will also be used as an estimation of $Q$ for actor training.

You should use all available rewards for value targets, so the formula for the value targets is simple:

$$
\hat v(s_t) = \sum_{t'=0}^{T - 1}\gamma^{t'}r_{t+t'} + \gamma^T \hat{v}(s_{t+T}),
$$

where $s_{t + T}$ is the latest observation of the environment.

Any callable could be passed to `EnvRunner` to be applied to each partial trajectory after it is collected. 
Thus, we can implement and use `ComputeValueTargets` callable. 

**Do not forget** to use `trajectory['dones']` flags to check if you need to add the value targets at the next step when 
computing value targets for the current step.

**Bonus (+0.5 pts):** implement [Generalized Advantage Estimation (GAE)](https://arxiv.org/pdf/1506.02438.pdf) instead; use $\lambda \approx 0.95$ or even closer to 1 in experiment. 

In [86]:

class ComputeValueTargets:
    def __init__(self, policy, gamma=0.99, lamb=0.95):
        self.policy = policy
        self.gamma = gamma
        self.lamb = lamb


    def __call__(self, trajectory, latest_observation):
        '''
        This method should modify trajectory inplace by adding 
        an item with key 'value_targets' to it
        
        input:
            trajectory - dict from runner
            latest_observation - last state, numpy, (num_envs x channels x width x height)
        '''
        # value_targets
       
        rewards = trajectory['rewards']
        n = len(rewards)
        dones = trajectory['dones']
        act = self.policy.act(latest_observation)
        g = [self.gamma * act['values'].detach().squeeze() + rewards[-1]]
        for i in range(1, n):
            masks = ~dones[-(i+1)]
            v_prev = torch.tensor(masks)*g[-1]
            g.append(self.gamma *  v_prev + rewards[-(i + 1)])
        trajectory['value_targets'] = torch.cat(g[::-1])

        # (-------------------------------------------)
        # advantages
        values = trajectory['values'].copy()
        values.append(act['values'])
        
        advantages = torch.zeros(n, nenvs)
        masks = [torch.from_numpy(~i) for i in dones]
        # compute the advantages using GAE
        gae = 0.0
        for t in reversed(range(n)):
            td_error = (
                torch.tensor(rewards[t]) + self.gamma * masks[t] * values[t + 1] - values[t]
            )
            gae = td_error + self.gamma * self.lamb * masks[t] * gae
            advantages[t] = gae
        trajectory['advantages'] = advantages.flatten()
        

In [87]:
trajectory.keys()

dict_keys(['actions', 'logits', 'log_probs', 'values', 'entropy', 'observations', 'rewards', 'dones'])

After computing value targets we will transform lists of interactions into tensors
with the first dimension `batch_size` which is equal to `T * nenvs`.

You need to make sure that after this transformation `"log_probs"`, `"value_targets"`, `"values"` are 1-dimensional PyTorch tensors.

In [90]:
class MergeTimeBatch:
    """ Merges first two axes typically representing time and env batch. """
    def __call__(self, trajectory, latest_observation):
        # Modify trajectory inplace. 
        for k, v in trajectory.items():
            if k not in ["value_targets", "advantages"]:
                trajectory[k] = torch.cat([torch.tensor(i) if not torch.is_tensor(i) else i for i in trajectory[k]])

Let's do more sanity checks!

In [91]:
runner = EnvRunner(env, policy, nsteps=5, transforms=[ComputeValueTargets(policy),
                                                      MergeTimeBatch()])

trajectory = runner.get_next()

In [92]:
# More sanity checks
assert 'value_targets' in trajectory, "Value targets not found"
assert trajectory['log_probs'].shape == (5 * nenvs,)
assert trajectory['value_targets'].shape == (5 * nenvs,)
assert trajectory['values'].shape == (5 * nenvs,)

assert trajectory['log_probs'].requires_grad, "Gradients are not available for actor head!"
assert trajectory['values'].requires_grad, "Gradients are not available for critic head!"

In [93]:
trajectory.keys()

dict_keys(['actions', 'logits', 'log_probs', 'values', 'entropy', 'observations', 'rewards', 'dones', 'value_targets', 'advantages'])

Now is the time to implement the advantage actor critic algorithm itself. You can look into [Mnih et al. 2016](https://arxiv.org/abs/1602.01783) paper, and lectures ([part 1](https://www.youtube.com/watch?v=Ds1trXd6pos&list=PLkFD6_40KJIwhWJpGazJ9VSj9CFMkb79A&index=5), [part 2](https://www.youtube.com/watch?v=EKqxumCuAAY&list=PLkFD6_40KJIwhWJpGazJ9VSj9CFMkb79A&index=6)) by Sergey Levine.

In [98]:
from collections import defaultdict
from torch.nn.utils import clip_grad_norm_

class A2C:
    def __init__(self, policy, optimizer, value_loss_coef=0.25, entropy_coef=0.01, max_grad_norm=0.5):
        self.policy = policy
        self.optimizer = optimizer
        self.value_loss_coef = value_loss_coef
        self.entropy_coef = entropy_coef
        self.max_grad_norm = max_grad_norm

        
    def loss(self, trajectory, write):
        # compute all losses
        # do not forget to use weights for critic loss and entropy loss
        
        values = trajectory['values']
        advantages = trajectory['advantages']
        value_targets = trajectory['value_targets']
        
        actions = trajectory['actions']
        logits = trajectory['logits']
        log_probs = trajectory['log_probs']
        
        
        policy_loss = -torch.mean(log_probs * advantages.detach())
        critic_loss = advantages.pow(2).mean()
        entropy_loss = -trajectory['entropy'].mean()
        
        # log all losses
        write('losses', {
            'policy loss': policy_loss,
            'critic loss': critic_loss,
            'entropy loss': entropy_loss
        })
        
        # additional logs
        write('critic/advantage', advantages[-1].item())
        write('critic/values', {
            'value predictions': values[-1].item(),
            'value targets':value_targets[-1].item(),
        })
        
        # return scalar loss
        return policy_loss + self.value_loss_coef * critic_loss + self.entropy_coef * entropy_loss         

    def train(self, runner):
        # collect trajectory using runner
        # compute loss and perform one step of gradient optimization
        # do not forget to clip gradients
        while True:
            trajectory = runner.get_next()
            
            loss = self.loss(trajectory, runner.write)
            self.optimizer.zero_grad()
            loss.backward()
           
            gradient_norm = clip_grad_norm_(runner.policy.model.parameters(), 
                                            self.max_grad_norm)
            self.optimizer.step()
            runner.write('gradient norm', gradient_norm)
                            

Now you can train your model. For optimization we suggest you use RMSProp with learning rate 7e-4 (you can also linearly decay it to 0), smoothing constant (alpha in PyTorch) equal to 0.99 and epsilon equal to 1e-5.

We recommend to train for at least 10 million environment steps across all batched environments (takes ~3 hours on a single GTX1080 with 8 CPU). It should be possible to achieve *average raw reward over last 100 episodes* (the average is taken over 100 last episodes in each environment in the batch) of about 600. **Your goal is to reach 500**.

Notes:
* if your reward is stuck at ~200 for more than 2M steps then probably there is a bug
* if your gradient norm is >10 something probably went wrong
* make sure your `entropy loss` is negative, your `critic loss` is positive
* make sure you didn't forget `.detach` in losses where it's needed
* `actor loss` should oscillate around zero or near it; do not expect loss to decrease in RL ;)
* you can experiment with `nsteps` ("rollout length"); standard rollout length is 5 or 10. Note that this parameter influences how many algorithm iterations is required to train on 10M steps (or 40M frames --- we used frameskip in preprocessing).

In [95]:
model = A2CNet(N_FRAMES_STACKED, n_actions)
policy = Policy(model)
runner = EnvRunner(env, policy, nsteps=10, transforms=[ComputeValueTargets(policy),
                                                      MergeTimeBatch()])

optimizer = torch.optim.RMSprop(policy.model.parameters(), 
                                lr=7e-04, 
                                alpha=0.99, 
                                eps=1e-05)

a2c = A2C(policy, optimizer)

In [18]:
#a2c.train(runner)
use_tensorboard = True
if use_tensorboard:
    %load_ext tensorboard
    %tensorboard --logdir logs

In [99]:
model = A2CNet(N_FRAMES_STACKED, n_actions)
model.load_state_dict(torch.load("A2C", weights_only=True))
policy = Policy(model)
runner = EnvRunner(env, policy, nsteps=10, transforms=[ComputeValueTargets(policy),
                                                      MergeTimeBatch()])

optimizer = torch.optim.RMSprop(policy.model.parameters(), 
                                lr=7e-04, 
                                alpha=0.99, 
                                eps=1e-05)

a2c = A2C(policy, optimizer)




In [100]:
a2c.train(runner)

KeyboardInterrupt: 

In [20]:
# save your model just in case 
torch.save(model.state_dict(), "A2C")    

In [21]:
env.close()

## Evaluation

In [37]:
env = nature_dqn_env("SpaceInvadersNoFrameskip-v4", clip_reward=False, episodic_life=False)

In [38]:
def evaluate(env, policy, n_games=1, t_max=10000):
    '''
    Plays n_games and returns rewards
    '''
    rewards = []
    
    for _ in range(n_games):
        s, info = env.reset()
        
        R = 0
        for _ in range(t_max):
            action = policy.act(np.array([s]))["actions"][0]
            
            s, r, term, trank, _ = env.step(action)
            
            R += r
            if term or trank:
                break

        rewards.append(R)
    return np.array(rewards)

In [41]:
# evaluation will take some time!
sessions = evaluate(env, policy, n_games=30)
score = sessions.mean()
print(f"Your score: {score}")

assert score >= 500, "Needs more training?"
print("Well done!")

Your score: 521.0
Well done!


In [42]:
env.close()

## Record

In [ ]:
env_monitor = nature_dqn_env("SpaceInvadersNoFrameskip-v4", monitor=True, clip_reward=False, episodic_life=False)

In [ ]:
# record sessions
sessions = evaluate(env_monitor, policy, n_games=3)

In [ ]:
# rewards for recorded games
sessions

In [ ]:
env_monitor.close()